# Phase 8: Cortex AI — LLM Functions in SQL

Use Snowflake's built-in AI functions to analyze ecommerce data. No ML setup — just SQL.

In [ ]:
%%sql -r product_ideas
SELECT AI_COMPLETE(
    'llama3.1-8b',
    'You are a data analyst. Given these product categories: Electronics, Footwear, Kitchen, Fitness, Office — suggest 3 new product ideas for each category. Be concise.'
) AS product_suggestions;

In [ ]:
%%sql -r classified_customers
SELECT
    CUSTOMER_NAME, COUNTRY,
    AI_CLASSIFY(COUNTRY, ['North America', 'Europe', 'Asia', 'South America', 'Middle East', 'Oceania', 'Africa']):label::VARCHAR AS REGION
FROM SALES_DW.STAGING.STG_CUSTOMERS
ORDER BY REGION;

In [ ]:
%%sql -r sentiment_results
WITH reviews AS (
    SELECT 1 AS ID, 'Wireless Headphones' AS PRODUCT, 'Amazing sound quality! Best headphones I ever bought.' AS REVIEW
    UNION ALL SELECT 2, 'Running Shoes', 'Terrible quality. Sole came off after 2 weeks.'
    UNION ALL SELECT 3, 'Coffee Maker', 'Decent product. Does what it says but nothing special.'
    UNION ALL SELECT 4, 'Yoga Mat', 'Love this mat! Perfect grip and super comfortable.'
    UNION ALL SELECT 5, 'Laptop Stand', 'Broke after a month. Cheap materials.'
)
SELECT ID, PRODUCT, REVIEW, AI_SENTIMENT(REVIEW) AS SENTIMENT
FROM reviews ORDER BY SENTIMENT:score DESC;

In [ ]:
%%sql -r extracted_tickets
WITH tickets AS (
    SELECT 1 AS ID, 'Hi, my name is John Smith. Order #1001 arrived damaged. Email: john@example.com' AS MESSAGE
    UNION ALL SELECT 2, 'Alice Johnson here, order 1006 for Bluetooth Speaker arrived late. Contact: alice@example.com'
)
SELECT ID, AI_EXTRACT(MESSAGE, ['customer_name', 'order_number', 'product', 'email']) AS EXTRACTED
FROM tickets;

In [ ]:
%%sql -r translated
SELECT
    'Thank you for your purchase! Your order has been shipped.' AS ORIGINAL,
    AI_TRANSLATE('Thank you for your purchase! Your order has been shipped.', 'en', 'es') AS SPANISH,
    AI_TRANSLATE('Thank you for your purchase! Your order has been shipped.', 'en', 'fr') AS FRENCH;

In [ ]:
%%sql -r fitness_products
SELECT PRODUCT_NAME, CATEGORY, BRAND, PRICE
FROM SALES_DW.STAGING.STG_PRODUCTS
WHERE AI_FILTER(PROMPT('Is this product related to working out or physical exercise? Product: {0}', PRODUCT_NAME));

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

summary_data = session.sql("""
    SELECT c.SEGMENT, COUNT(DISTINCT o.ORDER_ID) AS ORDERS,
           ROUND(SUM(o.TOTAL_AMOUNT), 2) AS REVENUE
    FROM SALES_DW.STAGING.STG_CUSTOMERS c
    JOIN SALES_DW.STAGING.STG_ORDERS o ON c.CUSTOMER_ID = o.CUSTOMER_ID
    GROUP BY c.SEGMENT ORDER BY REVENUE DESC
""").to_pandas()

insight = session.sql(f"""
    SELECT AI_COMPLETE('llama3.1-8b',
        'Analyze this data and give 3 insights and 2 recommendations. Be concise.\n\n{summary_data.to_string()}')
""").collect()[0][0]
print(insight)

In [ ]:
%%sql -r redacted_data
SELECT CUSTOMER_NAME, EMAIL,
    AI_REDACT('Customer ' || CUSTOMER_NAME || ' with email ' || EMAIL || ' from ' || COUNTRY) AS REDACTED
FROM SALES_DW.STAGING.STG_CUSTOMERS LIMIT 5;